# 🎙️ GPT-SoVITS 声音克隆 — 听书club 定制版

> 在 Google Colab 免费 GPU (T4) 上完成**声音克隆 + 语音合成**。中文效果最好的开源方案。
>
> **适用场景**：克隆听书club 主持人声线 → 批量生成读书音频（替代 edge-tts 机器人声）。
>
> ⚠️ 需科学上网访问 Colab。免费 T4 有每日时长限制（约12~24h），训练 1000 步足够。
>
> 流程：`准备参考音频 → 环境配置(一次) → 启动 WebUI → 零样本克隆 或 微调训练`

## 第 0 步：准备参考音频（关键！克隆效果取决于它）

**要求**：
- 3~10 秒 **干净人声**（零样本推理） / 1~5 分钟（微调训练，效果更好）
- 单人、无 BGM、无杂音、无重音口播，中文普通话
- 格式：wav / mp3 均可

**从已有 mp3 语料提取**（听书club 现有 120+ 本音频）：
1. 在电脑上执行（WSL 里也可）：
```bash
ffmpeg -y -ss 0 -t 15 -i "《某书》.mp3" -acodec pcm_s16le -ar 22050 -ac 1 ref_15s.wav
```
2. 把 `ref_15s.wav` 上传到 Colab（左侧文件面板拖拽，或下方代码运行时上传）
3. 或在 Colab 内用本 notebook 的「录制」功能（WebUI 支持直接录音）

> 参考音频越干净，克隆越像。开场白「大家好，歡迎來到《听书club》…」是最理想的素材。

In [ ]:
# 检查 GPU（必须为 True，且显存 ≥ 15GB 才适合训练；零样本推理 T4 完全够）
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 第 1 步：环境配置（只运行一次，约 15~25 分钟）

> 安装 Anaconda + GPT-SoVITS + 预训练模型（下载源 HF，Colab 海外网络快）。
> **本步骤只需执行一次**。之后每次使用只需「运行时 → 重新连接」并跳过本步。

In [ ]:
# 安装 condacolab（Colab 内的 Anaconda 环境管理）
%pip install -q condacolab
import condacolab
condacolab.install_from_url("https://repo.anaconda.com/archive/Anaconda3-2024.10-1-Linux-x86_64.sh")

# 注意：上面会自动重启运行时，重启后请手动回到这里继续执行下一格

In [ ]:
# 克隆仓库 + 创建 conda 环境 + 安装依赖 + 下载预训练模型
%%writefile /content/setup.sh
set -e
cd /content
if [ ! -d /content/GPT-SoVITS/.git ]; then
  git clone https://github.com/RVC-Boss/GPT-SoVITS.git
fi
cd GPT-SoVITS
if conda env list | awk '{print $1}' | grep -Fxq "GPTSoVITS"; then
    :
else
    conda create -n GPTSoVITS python=3.10 -y
fi
source activate GPTSoVITS
pip install ipykernel -q
bash install.sh --device CU126 --source HF --download-uvr5
echo "=== SETUP DONE ==="

!cd /content && bash setup.sh

## 第 2 步：启动 WebUI（每次使用都要运行）

> 启动后输出一个 **gradio.live 公网链接**，点击打开图形界面。
> WebUI 内含：参考音频上传 / 零样本克隆 / 微调训练 / 批量合成。

In [ ]:
# 后台启动 WebUI（约 1~3 分钟），自动抓取公网链接
import subprocess, time, re, os, sys

# 终止可能残留的旧进程
for p in ["webui.py", "api.py", "api_v2.py"]:
    subprocess.run(["pkill", "-f", p], capture_output=True)

log_path = "/content/webui.log"
with open(log_path, "w") as f:
    proc = subprocess.Popen(
        ["bash", "-lc", "cd /content/GPT-SoVITS && source activate GPTSoVITS && export is_share=True && python webui.py"],
        stdout=f, stderr=subprocess.STDOUT
    )

# 轮询日志找公网链接
url = None
for _ in range(180):
    time.sleep(2)
    try:
        log = open(log_path, encoding="utf-8", errors="ignore").read()
    except FileNotFoundError:
        continue
    m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", log)
    if m:
        url = m.group(0)
        break
    if "Traceback" in log and "Error" in log:
        print("⚠️ 启动报错，查看最后 30 行日志：")
        print("\n".join(log.strip().splitlines()[-30:]))
        break

if url:
    print("✅ WebUI 已启动！公网链接（请打开）：")
    print("\n" + "="*60)
    print(url)
    print("="*60)
    print("\n若链接打不开，等 10 秒后执行：!tail -50 /content/webui.log 查看新链接")
else:
    print("⏳ 链接尚未出现，查看日志：")
    print("\n".join(open(log_path, encoding="utf-8", errors="ignore").read().strip().splitlines()[-20:]))

## 第 3 步：使用指南

### A. 零样本克隆（最快，10 秒参考音频即可）
在 WebUI 打开后：
1. 顶部标签页进入 **1-GPT-SoVITS-TTS → 1C-推理**
2. 上传参考音频（第0步准备），填写参考音频的**文字内容**（重要！必须与音频一致）
3. 在「需要合成的文本」输入要生成的内容（如听书club 脚本段落）
4. 点击「合成语音」→ 试听 → 满意后下载

### B. 微调训练（推荐：克隆更像，音色更稳）
> 适合听书club：用主持人 1~5 分钟真实语料训练专属模型。
1. 标签页进入 **1-GPT-SoVITS-TTS → 1A-训练集格式化工具**
   - 上传 1~5 分钟参考音频 + 完整文字稿（WebUI 支持自动 ASR 标注）
   - 点击「开启处理」→ 输出路径如 `/content/GPT-SoVITS/output/asr_opt`
2. 进入 **1B-微调训练**
   - 实验名填 `tingshu_club`，点击「① 解析数据」→「② 训练 Sovits」→「③ 训练 GPT」
   - 训练 500~1000 步即可（T4 上约 30~60 分钟），步数越高越像但也越易过拟合
3. 回到 **1C-推理**，模型路径选训练出的 `tingshu_club_*` 模型，即可用专属音色合成

### C. 批量合成（听书club 长音频）
WebUI 推理页支持导入文本文件批量合成；或使用 API 模式（见下一格）自动分批合成 + ffmpeg 合并，与现有生产流水线衔接。

## 第 4 步（可选）：API 批量合成服务

启动 GPT-SoVITS 的 OpenAI 兼容 API 服务（端口 9880），配合 Python 脚本实现「长文本分段 → 批量合成 → ffmpeg 合并」，直接接入听书club 生产流水线。

In [ ]:
# 启动 API 服务（v2，OpenAI 兼容，端口 9880）
import subprocess, time, re

# 使用默认预训练模型；若训练了专属模型，替换 -s 和 -g 参数为 /content/GPT-SoVITS/pretrained/ 下对应文件
# 更推荐：用训练好的模型（在 1B 微调训练输出目录里，如 GPT_weights/tingshu_club_e8_s8.pth 和 SoVITS_weights/tingshu_club_e8_s8.pth）

# 下面先用预训练模型启动（零样本模式）
api_log = "/content/api.log"
with open(api_log, "w") as f:
    proc = subprocess.Popen(
        ["bash", "-lc",
         "cd /content/GPT-SoVITS && source activate GPTSoVITS && "
         "python api_v2.py -a 127.0.0.1 -p 9880 -c GPT_SoVITS/configs/tts_infer.yaml 2>&1"],
        stdout=f, stderr=subprocess.STDOUT
    )

# 等端口就绪
import socket
ready = False
for _ in range(120):
    time.sleep(2)
    s = socket.socket(); s.settimeout(1)
    if s.connect_ex(("127.0.0.1", 9880)) == 0:
        ready = True; s.close(); break
    s.close()

if ready:
    print("✅ API 服务已就绪 (127.0.0.1:9880)，支持 OpenAI /v1/audio/speech 接口")
else:
    print("⚠️ API 未就绪，日志：")
    print(open(api_log, encoding="utf-8", errors="ignore").read()[-1000:])

In [ ]:
# API 调用示例（零样本克隆：传参考音频 + 文本 → 返回合成音频）
import requests, json, base64

# 1) 上传参考音频，获得 reference_id（只需做一次）
#    把 /content/ 下的参考音频路径填进来
ref_audio = "/content/ref_15s.wav"  # 改成你上传的参考音频路径
ref_text  = "大家好，歡迎來到《听书club》，今天為你解讀的是"  # 改成参考音频的实际文字

with open(ref_audio, "rb") as f:
    base64_audio = base64.b64encode(f.read()).decode()

resp = requests.post("http://127.0.0.1:9880/change_refer", json={
    "refer_wav_path": ref_audio,
    "prompt_text": ref_text,
    "prompt_language": "zh"
}, timeout=60)
print("change_refer:", resp.json())

# 2) 合成文本（OpenAI 兼容格式）
text = "大家好，歡迎來到《听书club》。今天為你解讀的是《反脆弱》：那些杀不死我们的，终将使我们更强大。"
r = requests.post("http://127.0.0.1:9880/v1/audio/speech", json={
    "model": "GPT-SoVITS",
    "input": text,
    "voice": "default"
}, timeout=120)
print("HTTP:", r.status_code, "bytes:", len(r.content))
if r.status_code == 200:
    with open("/content/output_api.wav", "wb") as f:
        f.write(r.content)
    print("✅ 已保存 /content/output_api.wav （可在左侧文件面板下载）")

## 长文本批量合成（听书club 10,000 字脚本）

```python
import re, subprocess, requests

text = open("/content/script.txt", encoding="utf-8").read()
# 按句切分，每段 ≤ 300 字（保证合成稳定）
chunks, cur = [], ""
for s in re.split(r"(?<=[。！？\n])", text):
    if len(cur) + len(s) > 300: chunks.append(cur); cur = s
    else: cur += s
if cur: chunks.append(cur)

parts = []
for i, ch in enumerate(chunks):
    r = requests.post("http://127.0.0.1:9880/v1/audio/speech",
                      json={"model": "GPT-SoVITS", "input": ch, "voice": "default"}, timeout=180)
    if r.status_code == 200:
        p = f"/content/part_{i:03d}.wav"; open(p, "wb").write(r.content); parts.append(p)
        print(i, "OK", len(ch), "字")

# ffmpeg 合并（注意：每段单独合成会有停顿差异，可后续用暖声后处理统一）
with open("/content/concat.txt", "w") as f:
    for p in parts: f.write(f"file '{p}'\n")
subprocess.run(["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", "/content/concat.txt",
                "-c", "copy", "/content/final_clone.wav"])
print("✅ 完成：/content/final_clone.wav （" + str(len(parts)) + " 段）")
```

## ⚠️ 常见问题

- **参考音频文字必须准确**：零样本克隆的相似度 80% 取决于参考音频与文字的一致性，用 Whisper 转写后手动校对。
- **Colab 断连**：训练/合成时别关标签页；长时间空闲会断，用「代码执行程序 → 更改运行时类型」重连（环境保留在磁盘上，重连后只需跑第2步）。
- **模型保存**：训练好的模型在 `/content/GPT-SoVITS/GPT_weights/` 和 `SoVITS_weights/`，**Colab 会定期清空**，重要模型及时下载回本地备份。
- **声音不像**：换更干净的参考音频（无BGM无混响）、增加微调训练步数、参考音频用同性别同语速。
- **显存不足**：T4 16GB 训练 1000 步没问题；报 OOM 就把实验名换新的重训（历史权重占用显存）。

---
*Notebook 由 Hermes Agent 生成 · 基于 RVC-Boss/GPT-SoVITS 官方 Colab-WebUI.ipynb*